# Assignment 3: Plant Disease Detection
## Convolutional Neural Network for Image Classification

In [ ]:
# Install required packages
!pip install tensorflow numpy pandas matplotlib seaborn scikit-learn pillow

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

In [ ]:
# Dataset configuration
# IMPORTANT: Update this path to your dataset location
dataset_base = r'New Plant Diseases Dataset(Augmented)'
train_dir = os.path.join(dataset_base, 'train')
valid_dir = os.path.join(dataset_base, 'valid')

# Check if validation folder exists
if not os.path.exists(valid_dir):
    print("No separate validation folder found. Will split from training data.")
    valid_dir = None

# Image parameters
img_size = 64  # Smaller size for faster training
batch_size = 64

print(f"Image size: {img_size}x{img_size}")
print(f"Batch size: {batch_size}")

In [ ]:
# Prepare data generators
if valid_dir is None:
    # Split from training data
    train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )
    
    valid_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )
else:
    # Separate validation folder
    train_datagen = ImageDataGenerator(rescale=1./255)
    valid_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True
    )
    
    valid_generator = valid_datagen.flow_from_directory(
        valid_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )

num_classes = len(train_generator.class_indices)
class_names = list(train_generator.class_indices.keys())

print(f"\nNumber of classes: {num_classes}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {valid_generator.samples}")
print(f"\nClasses: {class_names}")

In [ ]:
# Define hyperparameter configurations to test
configs = [
    {'name': 'Config 1: ReLU, Batch 64', 'activation': 'relu', 'batch_size': 64},
    {'name': 'Config 2: ReLU, Batch 128', 'activation': 'relu', 'batch_size': 128},
    {'name': 'Config 3: Tanh, Batch 64', 'activation': 'tanh', 'batch_size': 64},
]

print("Testing 3 configurations:")
for i, config in enumerate(configs, 1):
    print(f"{i}. {config['name']}")

print("\nNote: Training will take 5-10 minutes per configuration")

In [ ]:
# Train models with different configurations
results = []

for config in configs:
    print(f"\n{'='*60}")
    print(f"Training: {config['name']}")
    print(f"{'='*60}")
    
    # Update batch size
    train_generator.batch_size = config['batch_size']
    valid_generator.batch_size = config['batch_size']
    
    # Build CNN model
    model = Sequential([
        # Conv Block 1
        Conv2D(32, 3, activation=config['activation'], input_shape=(img_size, img_size, 3)),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Conv Block 2
        Conv2D(64, 3, activation=config['activation']),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Conv Block 3
        Conv2D(128, 3, activation=config['activation']),
        BatchNormalization(),
        MaxPooling2D(2),
        
        # Dense layers
        Flatten(),
        Dense(128, activation=config['activation']),
        BatchNormalization(),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    
    # Compile model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    # Train model (only 3 epochs for speed)
    print("Training...")
    history = model.fit(
        train_generator,
        epochs=3,
        validation_data=valid_generator,
        verbose=1
    )
    
    # Get final metrics
    val_acc = history.history['val_accuracy'][-1]
    val_loss = history.history['val_loss'][-1]
    
    # Store results
    results.append({
        'Config': config['name'],
        'Activation': config['activation'],
        'Batch Size': config['batch_size'],
        'Val Accuracy': val_acc,
        'Val Loss': val_loss
    })
    
    print(f"✓ Validation Accuracy: {val_acc:.4f}")
    print(f"✓ Validation Loss: {val_loss:.4f}")

print("\n" + "="*60)
print("All configurations trained!")
print("="*60)

In [ ]:
# Display results table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Val Accuracy', ascending=False)
print("\nRESULTS TABLE (Sorted by Accuracy):")
print(results_df.to_string(index=False))

# Identify best model
best_config = results_df.iloc[0]
print(f"\n{'='*60}")
print("BEST MODEL:")
print(f"Configuration: {best_config['Config']}")
print(f"Accuracy: {best_config['Val Accuracy']:.4f} ({best_config['Val Accuracy']*100:.2f}%)")
print(f"Loss: {best_config['Val Loss']:.4f}")
print(f"{'='*60}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
axes[0].barh(results_df['Config'], results_df['Val Accuracy'], color='skyblue', edgecolor='black')
axes[0].set_xlabel('Validation Accuracy', fontweight='bold', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontweight='bold', fontsize=14)
axes[0].set_xlim(0, 1)
axes[0].invert_yaxis()
for i, v in enumerate(results_df['Val Accuracy']):
    axes[0].text(v, i, f' {v:.4f}', va='center')

# Loss comparison
axes[1].barh(results_df['Config'], results_df['Val Loss'], color='salmon', edgecolor='black')
axes[1].set_xlabel('Validation Loss', fontweight='bold', fontsize=12)
axes[1].set_title('Loss Comparison', fontweight='bold', fontsize=14)
axes[1].invert_yaxis()
for i, v in enumerate(results_df['Val Loss']):
    axes[1].text(v, i, f' {v:.4f}', va='center')

plt.tight_layout()
plt.savefig('assignment3_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("Visualization saved as 'assignment3_results.png'")

In [ ]:
# Train final best model
best_activation = best_config['Activation']
best_batch_size = int(best_config['Batch Size'])

print(f"Training final model with best configuration...")
print(f"Activation: {best_activation}, Batch Size: {best_batch_size}")

train_generator.batch_size = best_batch_size
valid_generator.batch_size = best_batch_size

final_model = Sequential([
    Conv2D(32, 3, activation=best_activation, input_shape=(img_size, img_size, 3)),
    BatchNormalization(),
    MaxPooling2D(2),
    Conv2D(64, 3, activation=best_activation),
    BatchNormalization(),
    MaxPooling2D(2),
    Conv2D(128, 3, activation=best_activation),
    BatchNormalization(),
    MaxPooling2D(2),
    Flatten(),
    Dense(128, activation=best_activation),
    BatchNormalization(),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

final_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
final_model.fit(train_generator, epochs=3, validation_data=valid_generator, verbose=0)

print("✓ Final model trained!")

In [ ]:
# Test with custom image (not from training/testing data)
print("\n" + "="*60)
print("TESTING WITH CUSTOM IMAGE")
print("="*60)

# IMPORTANT: Place your custom test image and update the path here
# The image should be a plant leaf (preferably with one of the diseases)
custom_image_path = 'custom_leaf.jpg'  # UPDATE THIS PATH

# Check if custom image exists
if os.path.exists(custom_image_path):
    # Load and preprocess image
    img = load_img(custom_image_path, target_size=(img_size, img_size))
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Display image
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Custom Test Image', fontweight='bold', fontsize=14)
    plt.show()
    
    # Predict
    predictions = final_model.predict(img_array, verbose=0)[0]
    predicted_class_idx = np.argmax(predictions)
    predicted_class = class_names[predicted_class_idx]
    confidence = predictions[predicted_class_idx]
    
    print(f"\n{'='*60}")
    print(f"PREDICTION: {predicted_class}")
    print(f"Confidence: {confidence*100:.2f}%")
    print(f"{'='*60}")
    
    # Show top 3 predictions
    print("\nTop 3 Predictions:")
    top_3_idx = np.argsort(predictions)[-3:][::-1]
    for i, idx in enumerate(top_3_idx, 1):
        print(f"{i}. {class_names[idx]}: {predictions[idx]*100:.2f}%")
    
else:
    print(f"\n⚠ Custom image not found at: {custom_image_path}")
    print("\nTo test with custom image:")
    print("1. Place a plant leaf image in the same folder as this notebook")
    print("2. Update 'custom_image_path' variable with your image filename")
    print("3. Re-run this cell")
    
    # Use a sample from validation set instead
    print("\n" + "="*60)
    print("Using sample from validation set instead:")
    print("="*60)
    
    # Get one batch from validation
    sample_images, sample_labels = next(valid_generator)
    sample_img = sample_images[0:1]
    sample_label = sample_labels[0]
    
    # Display image
    plt.figure(figsize=(6, 6))
    plt.imshow(sample_images[0])
    plt.axis('off')
    plt.title('Sample Validation Image', fontweight='bold', fontsize=14)
    plt.show()
    
    # Predict
    predictions = final_model.predict(sample_img, verbose=0)[0]
    predicted_class_idx = np.argmax(predictions)
    true_class_idx = np.argmax(sample_label)
    predicted_class = class_names[predicted_class_idx]
    true_class = class_names[true_class_idx]
    confidence = predictions[predicted_class_idx]
    
    print(f"\nTrue Label: {true_class}")
    print(f"Predicted: {predicted_class}")
    print(f"Confidence: {confidence*100:.2f}%")
    print(f"Match: {'✓ Correct' if predicted_class == true_class else '✗ Incorrect'}")

In [ ]:
# Save results
results_df.to_csv('assignment3_results.csv', index=False)
print("\nResults saved to 'assignment3_results.csv'")
print("\n✓ Assignment 3 Complete!")